In [0]:
%sql 
select current_user(), current_catalog(), current_database(), current_schema() current_Schema_name


current_user(),current_catalog(),current_database(),current_Schema_name
patel.riyas@northeastern.edu,workspace,default,default


In [0]:
data = [[2021, "test", "Albany", "M", 42]]
columns = ["Year", "First_Name", "County", "Sex", "Count"]

df_custom = spark.createDataFrame(
    data,
    schema="Year int, First_Name STRING, County STRING, Sex STRING, Count int"
)

#display(df1)  # The display() method is specific to Databricks notebooks and provides a richer visualization.
df_custom.show()  # The show() method is part of Apache Spark DataFrame API and provides basic visualization.


+----+----------+------+---+-----+
|Year|First_Name|County|Sex|Count|
+----+----------+------+---+-----+
|2021|      test|Albany|  M|   42|
+----+----------+------+---+-----+



In [0]:
catalog = "workspace"
schema = "damg7370"
volume = "datastore"
download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
file_name = "ny_health_data2.csv"
table_name = "ny_health_data"

path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema

print(path_table)     # Show the complete path
print(path_volume)    # Show the complete path


workspace.damg7370
/Volumes/workspace/damg7370/datastore


In [0]:
dbutils.fs.cp(f"{download_url}", f"{path_volume}/{file_name}")


True

In [0]:
df_csv = spark.read.csv(
    f"{path_volume}/{file_name}",
    header=True,
    inferSchema=True,
    sep=","
)

df_csv.printSchema()
#display(df_csv)


root
 |-- Year: integer (nullable = true)
 |-- First Name: string (nullable = true)
 |-- County: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Count: integer (nullable = true)



In [0]:
df_csv = df_csv.withColumnRenamed("First Name", "First_Name")
df_csv.printSchema()


root
 |-- Year: integer (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- County: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Count: integer (nullable = true)



In [0]:
df1 = df_csv
df = df1.union(df_csv)

display(df)
df.printSchema()

print(df1.count())
print(df.count())


Year,First_Name,County,Sex,Count
2022,OLIVIA,Albany,F,16
2022,AMELIA,Albany,F,15
2022,AVERY,Albany,F,12
2022,EMMA,Albany,F,11
2022,CHARLOTTE,Albany,F,11
2022,CHLOE,Albany,F,11
2022,SOPHIA,Albany,F,8
2022,CORA,Albany,F,8
2022,MIA,Albany,F,7
2022,LUNA,Albany,F,7


root
 |-- Year: integer (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- County: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Count: integer (nullable = true)

99116
198232


In [0]:
display(df.filter(df["Count"] < 50))


Year,First_Name,County,Sex,Count
2022,OLIVIA,Albany,F,16
2022,AMELIA,Albany,F,15
2022,AVERY,Albany,F,12
2022,EMMA,Albany,F,11
2022,CHARLOTTE,Albany,F,11
2022,CHLOE,Albany,F,11
2022,SOPHIA,Albany,F,8
2022,CORA,Albany,F,8
2022,MIA,Albany,F,7
2022,LUNA,Albany,F,7


In [0]:
display(df.where(df["Count"] > 50))

Year,First_Name,County,Sex,Count
2022,SOPHIA,Bronx,F,65
2022,LUNA,Bronx,F,65
2022,EMMA,Bronx,F,60
2022,ISABELLA,Bronx,F,55
2022,LIAM,Bronx,M,156
2022,NOAH,Bronx,M,132
2022,ETHAN,Bronx,M,84
2022,JACOB,Bronx,M,68
2022,JAYDEN,Bronx,M,65
2022,AMIR,Bronx,M,64


In [0]:
from pyspark.sql.functions import desc

display(df.select("First_Name", "Count").orderBy(desc("Count")))


First_Name,Count
DAVID,297
DAVID,297
DAVID,287
DAVID,287
ETHAN,273
ETHAN,273
DAVID,273
DAVID,273
DAVID,272
LIAM,272


In [0]:
subsetDF = df.filter(
    (df["Year"] == 2009) & 
    (df["Count"] > 100) & 
    (df["Sex"] == "F")
).select("First_Name", "County", "Count") \
 .orderBy(desc("Count"))

display(subsetDF)


First_Name,County,Count
LEAH,Kings,186
LEAH,Kings,186
SARAH,Kings,183
SARAH,Kings,183
RACHEL,Kings,170
RACHEL,Kings,170
ESTHER,Kings,168
ESTHER,Kings,168
ISABELLA,Queens,165
ISABELLA,Queens,165


In [0]:
df.write.mode("overwrite").saveAsTable(f"{path_table}.{table_name}")


In [0]:
df.write.format("json").mode("overwrite").save("/Volumes/workspace/damg7370/datastore/json_data")
df.write.format("parquet").mode("overwrite").save("/Volumes/workspace/damg7370/datastore/parquet_data")